In [1]:
import sys
sys.path.append("..")

In [2]:
#| default_exp tests

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import torch 
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm
from ddpm_backtest.data_loaders import NiftyRiskDataset,compute_regime_score,fit_garch_volatility,get_nifty_regime_data,prepare_dataloaders
from ddpm_backtest.noising_time import noisify,set_seed,split_context,cosine_beta_scheduler
from ddpm_backtest.diffusion_utils import TabularDDPM,EMA
from ddpm_backtest.models import FiLMLayer,f_net
from ddpm_backtest.sampling_utils import _reverse_diffusion,_build_market_feats,_sample_raw, calibrate_rescale_factor, build_quantile_map, apply_quantile_map, sample_residuals,generate_path_ensemble_garch, evaluate_predictive_risk,plot_pit_histogram, plot_var_timeseries

[*********************100%***********************]  1 of 1 completed

[*********************100%***********************]  1 of 1 completed



Shape: (2566, 35)

Regime counts:
regime
0    1787
1     779
Name: count, dtype: int64
MARKET_DIM=25, TIME_DIM=8, CONTEXT_DIM=33
Device: cpu | T=100
Device: cpu | T=100


In [4]:
risk_df = get_nifty_regime_data()
risk_df['regime_score'] = compute_regime_score(risk_df)
risk_df['regime_score'] = risk_df['regime_score'].fillna(0.5)
risk_df['regime'] = (risk_df['regime_score'] > 0.5).astype(int)
risk_df.drop(columns=['sma_200','Close'], inplace=True)
risk_df = fit_garch_volatility(risk_df)
n_total = len(risk_df)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed



Fitting GARCH(1,1) on training data...
                     Constant Mean - GARCH Model Results                      
Dep. Variable:          target_return   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -2619.48
Distribution:                  Normal   AIC:                           5246.95
Method:            Maximum Likelihood   BIC:                           5269.46
                                        No. Observations:                 2052
Date:                Sun, May 03 2026   Df Residuals:                     2051
Time:                        16:58:56   Df Model:                            1
                                Mean Model                                
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu             0.0776  1.

In [5]:
PAST_RETURN_COLS   = ([f"return_lag_{l}" for l in [1, 2, 3, 5, 10, 21]] +
                      [f"cum_return_{w}d" for w in [5, 10, 21, 63]])
RVOL_COLS          = ([f"rvol_{w}d" for w in [5, 10, 21, 63]] +
                      ["vol_ratio_5_21", "vol_ratio_21_63"])
ORIGINAL_CONT_COLS = ["realized_vol", "VIX", "RSI", "skew",
                       "kurtosis", "vol_spike", "RSI_velocity", "garch_vol"]
UNSCALED_CONT_COLS = ["regime_score",
                       "dow_sin", "dow_cos", "month_sin", "month_cos",
                       "dom_sin", "dom_cos", "quarter_sin", "quarter_cos"]
SCALED_CONT_COLS   = ORIGINAL_CONT_COLS + PAST_RETURN_COLS + RVOL_COLS

MARKET_DIM  = len(SCALED_CONT_COLS) + 1
TIME_DIM    = len(UNSCALED_CONT_COLS) - 1
CONTEXT_DIM = len(SCALED_CONT_COLS) + len(UNSCALED_CONT_COLS)
print(f"MARKET_DIM={MARKET_DIM}, TIME_DIM={TIME_DIM}, CONTEXT_DIM={CONTEXT_DIM}")

MARKET_DIM=25, TIME_DIM=8, CONTEXT_DIM=33


In [6]:
train_dl, val_dl, condition_df, scaler_x, scaler_cond, val_df = prepare_dataloaders(risk_df)
print(f"\nscaler_x mean={scaler_x.mean_[0]:.4f}  scale={scaler_x.scale_[0]:.4f}  (mean~0, scale~1)")

Train: 2,052  Val: 257  Test: 257

scaler_x mean=0.0591  scale=1.0063  (mean~0, scale~1)

scaler_x mean=0.0591  scale=1.0063  (mean~0, scale~1)


In [7]:
T          = 100
betas      = cosine_beta_scheduler(T)
alphas     = 1 - betas
alphas_bar = torch.cumprod(torch.from_numpy(alphas).float(), dim=0)
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
alphas_bar = alphas_bar.to(device)
print(f"Device: {device} | T={T}")

Device: cpu | T=100


In [8]:
diffusion_model = TabularDDPM(
    d_in=1, cond_in_classes=2,
    scaled_cont_dim=len(SCALED_CONT_COLS),
    fixed_market_dim=1, time_dim=TIME_DIM,
    t_dim=64, dropout=0.1,
).to(device)
print(f"Parameters: {sum(p.numel() for p in diffusion_model.parameters()):,}")

optimizer        = torch.optim.Adam(diffusion_model.parameters(), lr=3e-4, weight_decay=1e-4)
num_epochs       = 500
p_uncond         = 0.15
warmup_eps       = 10
patience         = 25
best_val_mean    = np.inf
patience_counter = 0
ema_shadow_best  = None

def lr_lambda(epoch):
    if epoch < warmup_eps:
        return (epoch + 1) / warmup_eps
    progress = (epoch - warmup_eps) / max(1, num_epochs - warmup_eps)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
ema       = EMA(diffusion_model)

set_seed(42)
for epoch in range(num_epochs):
    diffusion_model.train()
    train_loss, n_train = 0.0, 0
    for x0, c, ctx in tqdm(train_dl, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
        x0, c, ctx = x0.to(device), c.to(device), ctx.to(device)
        if x0.dim() == 1: x0 = x0.unsqueeze(1)
        mc, tf     = split_context(ctx)
        drop_mask  = torch.rand(c.shape[0], device=device) < p_uncond
        xt, t, eps = noisify(T, x0, alphas_bar)
        eps_pred   = diffusion_model(xt, c, mc, tf, t.float(), drop_mask)
        snr     = alphas_bar[t] / (1.0 - alphas_bar[t])
        weights = (snr / snr.mean()).clamp(0.1, 10.0).unsqueeze(1)
        loss    = (weights * F.smooth_l1_loss(eps_pred, eps, reduction="none")).mean()
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(diffusion_model.parameters(), 1.0)
        optimizer.step(); ema.update(diffusion_model)
        train_loss += loss.item(); n_train += 1

    diffusion_model.eval()
    val_loss, n_val = 0.0, 0
    with torch.no_grad():
        for x0v, cv, ctxv in val_dl:
            x0v, cv, ctxv = x0v.to(device), cv.to(device), ctxv.to(device)
            if x0v.dim() == 1: x0v = x0v.unsqueeze(1)
            mcv, tfv = split_context(ctxv)
            xtv, tv, epsv = noisify(T, x0v, alphas_bar)
            val_loss += F.smooth_l1_loss(
                diffusion_model(xtv, cv, mcv, tfv, tv.float()), epsv).item()
            n_val += 1

    val_mean = val_loss / n_val
    if val_mean < best_val_mean:
        best_val_mean    = val_mean
        patience_counter = 0
        torch.save(diffusion_model.state_dict(), "best_model.pt")
        ema_shadow_best  = {k: v.clone() for k, v in ema.shadow.items()}
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
    scheduler.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:03d} | Train {train_loss/n_train:.6f} | "
              f"Val {val_mean:.6f} | LR {scheduler.get_last_lr()[0]:.2e} | "
              f"Patience {patience_counter}/{patience}")

diffusion_model.load_state_dict(torch.load("best_model.pt"))
ema.shadow = ema_shadow_best
print(f"\nLoaded best model (val={best_val_mean:.6f})")

Parameters: 117,441


Epoch 010 | Train 0.245095 | Val 0.396967 | LR 3.00e-04 | Patience 5/25


Epoch 020 | Train 0.314262 | Val 0.337490 | LR 3.00e-04 | Patience 2/25


Epoch 030 | Train 0.267276 | Val 0.455206 | LR 2.99e-04 | Patience 1/25


Epoch 040 | Train 0.324842 | Val 0.586193 | LR 2.97e-04 | Patience 2/25


...

Epoch 110 | Train 0.260850 | Val 0.176270 | LR 2.70e-04 | Patience 13/25


Epoch 120 | Train 0.250654 | Val 0.182240 | LR 2.64e-04 | Patience 2/25


Epoch 130 | Train 0.295288 | Val 0.138577 | LR 2.58e-04 | Patience 12/25


Epoch 140 | Train 0.227034 | Val 0.506283 | LR 2.51e-04 | Patience 22/25


Early stopping at epoch 143

Loaded best model (val=0.133474)


In [9]:
quantile_map = build_quantile_map(risk_df.iloc[:int(0.80*n_total)])

In [10]:
set_seed(42)
ema.apply(diffusion_model)
rescale_factor, target_resid_std = calibrate_rescale_factor(
    condition_df, diffusion_model, alphas, betas,
    scaler_x, scaler_cond, alphas_bar,
    risk_df   = risk_df.iloc[:int(0.80*n_total)],   
    n_calib   = 2000,
    guidance_scale = 1.5,
    temperature    = 1.0,
    scaled_cont_cols=SCALED_CONT_COLS,
    unscaled_cont_cols=UNSCALED_CONT_COLS,
    T=T
)
ema.restore(diffusion_model)

5th target=-1.6243  generated=-2105.8586  factor_95=0.0076
1st target=-2.9005  generated=-3073.4841  factor_99=0.0056
Blended factor (40% VaR95 + 60% VaR99): 0.1000


In [12]:
set_seed(42)
ema.apply(diffusion_model)
test_resids = sample_residuals(
    condition_df.iloc[-1], 1000, diffusion_model, alphas, betas,
    scaler_x, scaler_cond, quantile_map=quantile_map,
    guidance_scale=1.5, alphas_bar_in=alphas_bar,
    scaled_cont_cols=SCALED_CONT_COLS,
    unscaled_cont_cols=UNSCALED_CONT_COLS,
    T=T
)
ema.restore(diffusion_model)

gv = condition_df.iloc[-1]["garch_vol"]
print(f"\nAfter rescaling:")
print(f"  resid std={test_resids.std():.4f}  (target {target_resid_std:.4f})")
print(f"  VaR95 generated: {np.percentile(test_resids * gv, 5):.5f}")
print(f"  VaR95 actual:    {np.percentile(condition_df['target_return'], 5):.5f}")


After rescaling:
  resid std=1.0137  (target 1.0219)
  VaR95 generated: -0.02601
  VaR95 actual:    -0.01391


In [ ]:
#| export
def generate_path_ensemble_garch_with_breach_scaling(
    condition_df, n_paths, model, alphas, betas,
    scaler_x, scaler_cond, quantile_map=None,
    guidance_scale=1.5, temperature=1.0,
    alphas_bar_in=None, breach_scale=1.2,   
    n_diffusion_steps=100, scaled_cont_cols=None, unscaled_cont_cols=None, T=100
):
    T_eval   = len(condition_df)
    paths    = np.zeros((T_eval, n_paths))
    actual   = condition_df["target_return"].values
    prev_breach = False   

    for row_idx, (_, row) in enumerate(tqdm(condition_df.iterrows(),
                                             total=T_eval)):
        residuals = sample_residuals(
            row, n_paths, model, alphas, betas, scaler_x, scaler_cond,
            quantile_map=quantile_map,
            guidance_scale=guidance_scale,
            alphas_bar_in=alphas_bar_in,
            temperature=temperature,
            scaled_cont_cols=scaled_cont_cols,
            unscaled_cont_cols=unscaled_cont_cols,
            T=T
        )
        garch_vol = row["garch_vol"]
        effective_scale = breach_scale if prev_breach else 1.0
        paths[row_idx] = residuals * garch_vol * effective_scale
        var_95_today = np.percentile(paths[row_idx], 5)
        prev_breach  = actual[row_idx] < var_95_today

    actual = condition_df["target_return"].values
    var_95  = np.percentile(paths, 5,  axis=1)
    var_99  = np.percentile(paths, 1,  axis=1)
    es_95   = np.array([
        paths[i][paths[i] < var_95[i]].mean()
        if (paths[i] < var_95[i]).any() else var_95[i]
        for i in range(T_eval)
    ])
    es_99   = np.array([
        paths[i][paths[i] < var_99[i]].mean()
        if (paths[i] < var_99[i]).any() else var_99[i]
        for i in range(T_eval)
    ])

    print(f"\nGARCH-DDPM Ensemble with breach scaling ({T_eval} steps):")
    print(f"  VaR 95% breach rate: {(actual < var_95).mean():.3f}  (target 0.050)")
    print(f"  VaR 99% breach rate: {(actual < var_99).mean():.3f}  (target 0.010)")
    return paths, var_95, var_99, es_95, es_99

In [ ]:
set_seed(42)
actual_returns = condition_df["target_return"].dropna().values

ema.apply(diffusion_model)
paths, var_95, var_99, es_95, es_99 = generate_path_ensemble_garch_with_breach_scaling(
    condition_df   = condition_df,
    n_paths        = 10000,
    model          = diffusion_model,
    alphas         = alphas,
    betas          = betas,
    scaler_x       = scaler_x,
    scaler_cond    = scaler_cond,
    quantile_map   = quantile_map,
    guidance_scale = 1.0,
    temperature    = 0.9,
    alphas_bar_in  = alphas_bar,
    scaled_cont_cols=SCALED_CONT_COLS,
    unscaled_cont_cols=UNSCALED_CONT_COLS,
    T=T
)
ema.restore(diffusion_model)

var_results = evaluate_predictive_risk(paths, actual_returns)
pit_vals    = plot_pit_histogram(paths, actual_returns)
plot_var_timeseries(actual_returns, var_results, dates=condition_df.index.values)

In [18]:
#| export
from scipy import stats
def christoffersen_test(actual_returns,var,alpha=0.05):
  hits = (actual_returns < var).astype(int)
  T = len(hits)
  N = hits.sum()
  prev = hits[:-1]
  curr = hits[1:]
  n_00 = ((prev == 0) & (curr == 0)).sum()
  n_01 = ((prev == 0) & (curr == 1)).sum()
  n_10 = ((prev == 1) & (curr == 0)).sum()
  n_11 = ((prev == 1) & (curr == 1)).sum()
  pi_01 = n_01 / (n_00 + n_01) if (n_00  + n_01) > 0 else 0.0
  pi_11 = n_11 / (n_10 + n_11) if(n_10 + n_11) > 0 else 0.0
  pi = N/T
  print(f"\nTransition probabilities:")
  print(f"  π_01={pi_01:.4f}  (P(breach | no breach yesterday))")
  print(f"  π_11={pi_11:.4f}  (P(breach | breach yesterday))")
  print(f"  π   ={pi:.4f}   (unconditional breach rate, target={alpha})")

  if N == 0 or N == T:
    lr_pof = -2 * (T * np.log(1-alpha) + 0 if N == 0 else T * np.log(alpha))
  else:
    lr_pof = -2 * (N * np.log(alpha) + (T - N)*np.log(1-alpha) - N *np.log(pi) -  (T - N) * np.log(1 - pi))
  def safe_log(x):
    return np.log(x) if x > 0 else 0.0
  l1_h0 =  ((n_00 + n_10)*safe_log(1-pi) + (n_01 + n_11)*safe_log(pi))
  ll_h1 = (
        n_00 * safe_log(1 - pi_01) +
        n_01 * safe_log(pi_01)     +
        n_10 * safe_log(1 - pi_11) +
        n_11 * safe_log(pi_11)
    )



  lr_ind = -2 * (l1_h0 - ll_h1)
  lr_cc = lr_pof + lr_ind
  p_pof = 1 - stats.chi2.cdf(lr_pof, df=1)
  p_ind = 1 - stats.chi2.cdf(lr_ind, df=1)
  p_cc  = 1 - stats.chi2.cdf(lr_cc,  df=2)
  print(f"\n{'='*55}")
  print(f" CHRISTOFFERSEN CONDITIONAL COVERAGE TEST")
  print(f"{'='*55}")
  print(f"  {'Test':<30} {'Stat':>8} {'df':>4} {'p-val':>8} {'Status':>8}")
  print(f"  {'-'*55}")

  for name, stat, df, pval in [
      ("Kupiec POF (coverage)",    lr_pof, 1, p_pof),
      ("Independence (clustering)",lr_ind, 1, p_ind),
      ("Conditional Coverage (CC)",lr_cc,  2, p_cc),
  ]:
    status = "PASS ✓" if pval >= 0.05 else "FAIL ✗"
    print(f"  {name:<30} {stat:>8.4f} {df:>4} {pval:>8.4f} {status:>8}")

  print(f"\n  Clustering ratio π_11/π_01: "
          f"{pi_11/pi_01:.3f}  (1.0 = no clustering, >2.0 = strong clustering)"
          if pi_01 > 0 else "  Cannot compute ratio (π_01=0)")

  return {
        "hits": hits, "T": T, "N": N,
        "n_00": n_00, "n_01": n_01, "n_10": n_10, "n_11": n_11,
        "pi_01": pi_01, "pi_11": pi_11, "pi": pi,
        "lr_pof": lr_pof, "lr_ind": lr_ind, "lr_cc": lr_cc,
        "p_pof": p_pof, "p_ind": p_ind, "p_cc": p_cc,
    }


In [19]:
#| export
import statsmodels.api as sm
import pandas as pd
def run_dqa(h,lags=4,var=None):
  df = pd.DataFrame({'hit': h})
  for i in range(1,lags + 1):
    df[f'lag_{i}'] = df['hit'].shift(i)
  if var is not None:
    df['var'] = var
  df = df.dropna()
  y = df['hit']
  X = df.drop(columns=['hit'])
  X = sm.add_constant(X)
  model = sm.OLS(y,X)
  results = model.fit()
  terms_to_test = ['const = 0'] + [f'lag_{i} = 0' for i in range(1,lags  + 1)]
  if var is not None:
    terms_to_test.append('var = 0')
  hypothesis = ', '.join(terms_to_test)
  wald_test = results.wald_test(hypothesis,scalar=True)
  p_value = float(wald_test.pvalue)
  return {
        'Test Statistic': round(float(wald_test.statistic), 4),
        'p-value': round(p_value, 6),
        'Reject Null': p_value < 0.05,
        'Regression Summary': results.summary()
    }

In [ ]:
print(christoffersen_test(actual_returns,var_95,alpha=0.05))

In [ ]:
hit_list = (actual_returns < var_99).astype(int)
print(run_dqa(hit_list,lags=4,var=var_99))